# wandb-config-into-args — worked example 2: Filter wandb metadata keys when applying sweep config

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `wandb-config-into-args`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When `dict(wandb.config)` is called inside a sweep run, it includes both your hyperparameters and wandb-internal metadata entries (keys starting with `_`). The safe pattern is to use `hasattr(args, k)` as a filter before calling `setattr`, which naturally excludes any key not declared as a field on the args dataclass.

## Worked solution

**Step 1 — why wandb adds extra keys.**
wandb stores internal run metadata (`_wandb`, `_runtime`, `_step`) in the same config namespace as your hyperparameters. When you call `dict(wandb.config)`, all of these appear in the dict. If you naively call `setattr(args, k, v)` for every key, you'll get `AttributeError` on the underscore-prefixed ones.

**Step 2 — the hasattr guard.**
The pattern `if hasattr(args, k): setattr(args, k, v)` does two things: it checks the field exists before writing, and it implicitly skips metadata keys since they don't appear as dataclass fields. This is more robust than filtering by prefix because it also handles future wandb metadata keys automatically.

**Step 3 — mutation contract.**
The function mutates `args` in place AND returns it. This matches the ARENA convention where the sweep agent's `train()` function receives `args` by value and the caller expects it returned.

In [ ]:
import sys
from unittest.mock import MagicMock
from dataclasses import dataclass

sys.modules.setdefault('wandb', MagicMock())

@dataclass
class SweepArgs:
    learning_rate: float = 3e-4
    weight_decay: float = 0.01
    hidden_dim: int = 256
    wandb_project: str = 'sweep-demo'

def update_args_safe(args, config_dict):
    """Apply sweep config dict to dataclass args, skipping unknown keys."""
    for k, v in config_dict.items():
        if hasattr(args, k):
            setattr(args, k, v)
    return args

# Exercise it
args = SweepArgs()
print('Defaults:', args)

# Simulate what dict(wandb.config) returns during a sweep
config = {
    'learning_rate': 1e-4,
    'weight_decay': 0.001,
    '_wandb': {'is_jupyter_run': True},  # internal — must be skipped
    '_runtime': 42,                       # internal — must be skipped
    'unknown_future_key': 99,             # not on our dataclass
}
result = update_args_safe(args, config)
print('Updated:', result)
assert result.learning_rate == 1e-4
assert result.weight_decay == 0.001
print('Test passed: metadata keys skipped cleanly.')